# Gold layer (PySpark)

Gold layer for the Kolumbus/Entur bus project, the PySpark version of `build_gold.py`.
Spark reads the cleaned silver tables, builds every gold table, and checks each one against its source.

**Input**
- silver (cleaned): `alerts`, `routes`, `stops`, `stop_visits`, `vehicle_positions` (`<name>.parquet`)
- v6 outputs (optional): `stop_gps_status.parquet`, `time_lost_null.parquet`

**Output:** `<GOLD_DIR>/<table>.parquet` for every table (a Spark folder of part files; pandas,
DuckDB and Spark all read it as one table), plus `<GOLD_DIR>/gold.duckdb` with all of them for the dashboard.

**Design rules**
- Cubes store **additive** measures only (counts and sums). The dashboard filters, SUMs, then divides:
  `SELECT SUM(n_on_time)*1.0/SUM(n_valid) AS otp FROM gold_visits_cube WHERE line_ref IN (...) AND hour_band='am_peak'`.
  Averages or percentages stored per row would be wrong after a filter.
- Delay percentiles are not additive: the cubes carry histogram buckets (see `gold_buckets`); for exact
  percentiles use `fact_stop_visits` (slim row-level table).
- ET times in this feed behave like **departure** times (v6 finding), so every delay here is a departure delay.
- All timestamps in the output are UTC. Hours, weekdays and bands are Europe/Oslo local time, computed here
  (the silver `hour` and `weekday` columns are only compared, never used).
- Distinct counts (vehicles) are not additive: see `gold_vehicle_hour` for `COUNT(DISTINCT)` on demand.

Every table is reconciled against its source at the end (PASS/FAIL list, also stored in `gold_meta`).

**Run:** set the three folders in the next cell (or the environment variables `SILVER_DIR`, `V6_DIR`,
`GOLD_DIR`), then *Run All*. Needs `pyspark` and Java; `duckdb` only for the optional `gold.duckdb` file.

In [1]:
import json
import os
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

# ---------------- folders ----------------
SILVER_DIR = os.environ.get("SILVER_DIR", ".")       # folder with the cleaned silver parquet files
V6_DIR = os.environ.get("V6_DIR", "clean")           # folder with the v6 outputs (optional)
GOLD_DIR = os.environ.get("GOLD_DIR", "gold")        # output folder

# ---------------- settings ----------------
LOCAL_TZ = "Europe/Oslo"
EARLY_S, LATE_S = -60, 300                      # on time = EARLY_S <= delay < LATE_S (seconds, departure delay)
DELAY_EDGES = [-120, -60, 0, 60, 180, 300, 600]  # histogram edges (s); EARLY_S and LATE_S must be among them
TL_EDGES = [0, 5, 10, 20, 40]                   # time_lost_s histogram edges (s), from v6
HOUR_BANDS = {"night": (0, 5), "am_peak": (6, 8), "midday": (9, 14), "pm_peak": (15, 17), "evening": (18, 23)}
GRID_DEG = 0.002                                # position grid, about 220 m x 115 m at this latitude
SEG_MAX_S = 1800                                # ignore stop-to-stop times above this
HEADWAY_MIN_S, HEADWAY_MAX_S = 120, 3600        # scheduled headways outside this are not analysed
BUNCH_RATIO, GAP_RATIO = 0.5, 1.5               # bunched: actual < 0.5 x scheduled; large gap: actual > 1.5 x scheduled
ALERT_MIN_WINDOW_S, ALERT_MAX_WINDOW_S = 3600, 6 * 3600
OFFSET_CLIP_S = 300                             # ET-vs-GPS offsets beyond this are not averaged
assert EARLY_S in DELAY_EDGES and LATE_S in DELAY_EDGES, "OTP limits must be histogram edges"

BAND_OF_HOUR = {h: b for b, (lo, hi) in HOUR_BANDS.items() for h in range(lo, hi + 1)}
OUT = Path(GOLD_DIR)
OUT.mkdir(parents=True, exist_ok=True)

## Spark session
The session time zone is UTC, so timestamps are read and written as UTC; local time is computed
explicitly with `from_utc_timestamp(..., Europe/Oslo)`. ANSI mode is off so that values that cannot be
converted become null (like pandas `errors="coerce"`) instead of stopping the job.
pandas writes timestamps in nanoseconds, which Spark cannot read as timestamps; `nanosAsLong` reads them
as numbers instead, and `to_utc()` below turns them back into timestamps.

In [3]:
spark = (SparkSession.builder
         .appName("buspulse-gold")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.legacy.parquet.nanosAsLong", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

c:\Users\Hameed Computers\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0


## Helpers

In [4]:
CHECKS = []      # reconciliation results
TABLES = {}      # gold table -> row count


def check(label, ok, detail=""):
    CHECKS.append({"check": label, "ok": bool(ok), "detail": str(detail)})
    print(f"{'PASS' if ok else 'FAIL'}  {label}  {detail}")


def read(folder, name, required=True):
    p = Path(folder) / f"{name}.parquet"
    if not p.exists():
        if required:
            raise FileNotFoundError(f"missing {p}")
        print(f"note: {p} not found, skipping what depends on it")
        return None
    return spark.read.parquet(str(p))


def dtype(df, name):
    return dict(df.dtypes).get(name)


def opt(df, name, default=None):
    """The column if it exists, otherwise a constant."""
    return F.col(name) if name in df.columns else F.lit(default)


def to_bool(df, name):
    """true/1/t/yes (any case) -> True, everything else (incl. missing) -> False."""
    if name not in df.columns:
        return F.lit(False)
    if dtype(df, name) == "boolean":
        return F.coalesce(F.col(name), F.lit(False))
    s = F.lower(F.trim(F.col(name).cast("string")))
    return F.coalesce(s.isin("true", "1", "t", "yes"), F.lit(False))


def to_utc(df, name):
    """Timestamp column (UTC). Strings are parsed as ISO 8601; unparseable values become null."""
    if name not in df.columns:
        return F.lit(None).cast("timestamp")
    if dtype(df, name) in ("timestamp", "timestamp_ntz", "date"):
        return F.col(name).cast("timestamp")        # naive values are taken as UTC (session time zone)
    if dtype(df, name) == "bigint":                 # pandas nanosecond timestamp, read as a number
        return F.timestamp_micros(F.floor(F.col(name) / 1000).cast("long"))
    return F.expr(f"try_cast(`{name}` AS timestamp)")


def num(df, name):
    """Numeric column as double; missing column, unparseable values and NaN become null."""
    if name not in df.columns:
        return F.lit(None).cast("double")
    c = F.col(name).cast("double")
    return F.when(F.isnan(c), None).otherwise(c)


def label(df, name, also_empty=False):
    """Text column with null / 'None' / 'nan' (and optionally '') replaced by 'unknown'."""
    if name not in df.columns:
        return F.lit("unknown")
    c = F.col(name).cast("string")
    bad = ["None", "nan"] + ([""] if also_empty else [])
    return F.when(c.isNull() | c.isin(*bad), "unknown").otherwise(c)


def local(col_name):
    return F.from_utc_timestamp(F.col(col_name), LOCAL_TZ)


def weekday_mon0(ts):
    """0 = Monday ... 6 = Sunday (Spark's dayofweek is 1 = Sunday ... 7 = Saturday)."""
    return (F.dayofweek(ts) + 5) % 7


def hour_band(hour_col):
    band = F.lit("unknown")
    for b, (lo, hi) in reversed(list(HOUR_BANDS.items())):
        band = F.when(hour_col.between(lo, hi), b).otherwise(band)
    return band


def hist_cols(prefix, column, cond, edges):
    parts = []
    for i in range(len(edges) + 1):
        if i == 0:
            c = f"{column} < {edges[0]}"
        elif i == len(edges):
            c = f"{column} >= {edges[-1]}"
        else:
            c = f"{column} >= {edges[i - 1]} AND {column} < {edges[i]}"
        parts.append(f"SUM(CAST(({cond} AND {c}) AS INT)) AS {prefix}{i}")
    return ",\n  ".join(parts)


def bucket_table(metric, prefix, edges):
    rows = []
    for i in range(len(edges) + 1):
        lo = None if i == 0 else edges[i - 1]
        hi = None if i == len(edges) else edges[i]
        lbl = f"< {hi}" if lo is None else (f">= {lo}" if hi is None else f"{lo} to {hi}")
        rows.append((metric, f"{prefix}{i}", lo, hi, lbl))
    return rows


def emit(name, df, one_file=True):
    """Write a gold table to <GOLD_DIR>/<name>.parquet and register it as a SQL view.
    Timestamps are written as naive UTC (timestamp_ntz), as in the pandas version."""
    for c, t in df.dtypes:
        if t == "timestamp":
            df = df.withColumn(c, F.col(c).cast("timestamp_ntz"))
    path = str(OUT / f"{name}.parquet")
    (df.coalesce(1) if one_file else df).write.mode("overwrite").parquet(path)
    t = spark.read.parquet(path)
    t.createOrReplaceTempView(name)
    TABLES[name] = t.count()
    print(f"  {name}: {TABLES[name]:,} rows")
    return t


def q(sql):
    return spark.sql(sql)


def one(sql):
    return q(sql).first()[0]

## Staging: stop_visits

In [8]:
sv_raw = read(SILVER_DIR, "clean_stop_visits")
sv = sv_raw.withColumns({
    "aimed_time": to_utc(sv_raw, "aimed_time"),
    "observed_time": to_utc(sv_raw, "observed_time"),
    "is_actual": to_bool(sv_raw, "is_actual"),
    "cancelled": to_bool(sv_raw, "cancelled"),
    "no_prediction": to_bool(sv_raw, "no_prediction"),
    "delay_outlier": to_bool(sv_raw, "delay_outlier"),
    "delay_s": num(sv_raw, "delay_s"),
    "line_ref": label(sv_raw, "line_ref"),
    "direction_ref": label(sv_raw, "direction_ref"),
    "operator_ref": label(sv_raw, "operator_ref"),
    "journey_ref": label(sv_raw, "journey_ref"),
})
loc = local("aimed_time")
sv = sv.withColumns({
    "local_hour": F.coalesce(F.hour(loc), F.lit(-1)),
    "weekday": F.coalesce(weekday_mon0(loc), F.lit(-1)),            # 0 = Monday
    "service_date": (F.col("service_date").cast("string") if "service_date" in sv_raw.columns
                     else F.date_format(loc, "yyyy-MM-dd")),
})
sv = sv.withColumns({
    "hour_band": hour_band(F.col("local_hour")),
    "valid": F.col("is_actual") & ~F.col("cancelled") & ~F.col("delay_outlier") & F.col("delay_s").isNotNull(),
}).cache()
sv.createOrReplaceTempView("sv")

n_sv = sv.count()
dates = sorted(r[0] for r in sv.select("service_date").distinct().collect() if r[0] is not None)
print(f"stop_visits: {n_sv:,} rows, service dates {dates}")

stop_visits: 87,279 rows, service dates ['2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27']


## Sanity checks on the silver data

In [9]:
d_calc = F.col("observed_time").cast("double") - F.col("aimed_time").cast("double")
m = (F.col("is_actual") & F.col("observed_time").isNotNull() & F.col("aimed_time").isNotNull()
     & F.col("delay_s").isNotNull())
r = sv.where(m).agg(
    F.avg((F.abs(d_calc - F.col("delay_s")) <= 1).cast("int")).alias("agree"),
    F.avg((F.abs(d_calc + F.col("delay_s")) <= 1).cast("int")).alias("flipped")).first()
agree = r["agree"] if r["agree"] is not None else float("nan")
flipped = r["flipped"] if r["flipped"] is not None else float("nan")
check("delay_s equals observed_time - aimed_time (positive = late)", agree >= 0.9,
      f"agree {agree:.1%}, sign flipped {flipped:.1%}")

if "hour" in sv_raw.columns:
    h = F.expr("try_cast(hour AS double)")
    r = sv.agg(F.avg(F.coalesce(h == F.col("local_hour"), F.lit(False)).cast("int")).alias("loc"),
               F.avg(F.coalesce(h == F.hour("aimed_time"), F.lit(False)).cast("int")).alias("utc")).first()
    print(f"      silver `hour` equals local hour in {r['loc']:.1%}, UTC hour in {r['utc']:.1%} (this notebook always uses local)")

FAIL  delay_s equals observed_time - aimed_time (positive = late)  agree nan%, sign flipped nan%


## Dimensions

In [10]:
print("building tables")
emit("dim_line", read(SILVER_DIR, "routes").dropDuplicates(["line_ref"]))
emit("dim_stop", read(SILVER_DIR, "stops").dropDuplicates(["stop_ref"]))

d = F.to_date("service_date")
emit("dim_date", spark.createDataFrame([(x,) for x in dates], "service_date string")
     .withColumns({"weekday": weekday_mon0(d), "weekday_name": F.date_format(d, "EEEE")})
     .withColumn("is_weekend", F.col("weekday") >= 5)
     .orderBy("service_date"))
emit("dim_hour_band", spark.createDataFrame(sorted(BAND_OF_HOUR.items()), "local_hour int, hour_band string"))
emit("gold_buckets", spark.createDataFrame(
    bucket_table("delay_s", "d", DELAY_EDGES) + bucket_table("time_lost_s", "tl", TL_EDGES),
    "metric string, bucket string, lo int, hi int, label string"))

building tables


FileNotFoundError: missing routes.parquet

## Facts from stop_visits
The additive measures shared by the cubes: counts, sums, sums of squares (for standard deviations) and the
delay histogram.

In [ ]:
HIST = hist_cols("d", "delay_s", "valid", DELAY_EDGES)
MEASURES = f"""
  COUNT(*) AS n_calls,
  SUM(CAST(cancelled AS INT)) AS n_cancelled,
  SUM(CAST((no_prediction AND NOT cancelled) AS INT)) AS n_no_prediction,
  SUM(CAST((is_actual AND NOT cancelled) AS INT)) AS n_actual,
  SUM(CAST((is_actual AND NOT cancelled AND delay_outlier) AS INT)) AS n_outlier,
  SUM(CAST(valid AS INT)) AS n_valid,
  SUM(CAST((valid AND delay_s < {EARLY_S}) AS INT)) AS n_early,
  SUM(CAST((valid AND delay_s >= {EARLY_S} AND delay_s < {LATE_S}) AS INT)) AS n_on_time,
  SUM(CAST((valid AND delay_s >= {LATE_S}) AS INT)) AS n_late,
  COALESCE(SUM(CASE WHEN valid THEN delay_s END), 0) AS sum_delay_s,
  COALESCE(SUM(CASE WHEN valid THEN delay_s * delay_s END), 0) AS sum_delay_sq,
  {HIST}"""

cube_dims = "service_date, local_hour, hour_band, weekday, line_ref, direction_ref, operator_ref"
emit("gold_visits_cube", q(f"SELECT {cube_dims}, {MEASURES} FROM sv GROUP BY {cube_dims}"))
emit("gold_stop_cube", q(f"""SELECT service_date, hour_band, line_ref, stop_ref, {MEASURES} FROM sv
                             GROUP BY service_date, hour_band, line_ref, stop_ref"""))
emit("gold_stop_order_delay", q(f"""
    SELECT service_date, hour_band, line_ref, direction_ref, stop_order,
      COUNT(*) AS n_calls, SUM(CAST(valid AS INT)) AS n_valid,
      SUM(CAST((valid AND delay_s < {EARLY_S}) AS INT)) AS n_early,
      SUM(CAST((valid AND delay_s >= {LATE_S}) AS INT)) AS n_late,
      COALESCE(SUM(CASE WHEN valid THEN delay_s END), 0) AS sum_delay_s,
      COALESCE(SUM(CASE WHEN valid THEN delay_s * delay_s END), 0) AS sum_delay_sq
    FROM sv GROUP BY service_date, hour_band, line_ref, direction_ref, stop_order"""))
emit("fact_stop_visits", sv.select(
    "service_date", "local_hour", "hour_band", "weekday", "journey_ref", "line_ref", "direction_ref",
    "operator_ref", "stop_order", "stop_ref", "aimed_time", "observed_time", "delay_s", "valid",
    "cancelled", "no_prediction", "delay_outlier"), one_file=False)

## Segment run times and headways
- **Segment:** consecutive stops of one journey; departure-to-departure time = driving + dwell at the second stop.
- **Headway:** time between consecutive scheduled calls of one line at one stop and direction.

In [ ]:
SECS = "(unix_micros({a}) - unix_micros({b})) / 1e6"      # difference of two timestamps in seconds

emit("gold_segment_runtime", q(f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(stop_ref) OVER w AS from_stop_ref, LAG(stop_order) OVER w AS prev_order,
                 LAG(aimed_time) OVER w AS prev_aimed, LAG(observed_time) OVER w AS prev_obs
          FROM v WINDOW w AS (PARTITION BY service_date, journey_ref ORDER BY stop_order)),
    s AS (SELECT *, {SECS.format(a='observed_time', b='prev_obs')} AS act_s,
                    {SECS.format(a='aimed_time', b='prev_aimed')} AS aim_s
          FROM p WHERE prev_order = stop_order - 1)
    SELECT service_date, hour_band, line_ref, from_stop_ref, stop_ref AS to_stop_ref,
           COUNT(*) AS n, SUM(act_s) AS sum_actual_s, SUM(aim_s) AS sum_aimed_s, SUM(act_s * act_s) AS sum_actual_sq
    FROM s WHERE act_s BETWEEN 0 AND {SEG_MAX_S} AND aim_s > 0
    GROUP BY service_date, hour_band, line_ref, from_stop_ref, stop_ref"""))

emit("gold_headway", q(f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(aimed_time) OVER w AS pa, LAG(observed_time) OVER w AS po
          FROM v WINDOW w AS (PARTITION BY service_date, line_ref, direction_ref, stop_ref ORDER BY aimed_time)),
    h AS (SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
                 {SECS.format(a='aimed_time', b='pa')} AS aim_h, {SECS.format(a='observed_time', b='po')} AS act_h
          FROM p WHERE pa IS NOT NULL)
    SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
           COUNT(*) AS n_pairs, SUM(act_h) AS sum_actual_headway_s, SUM(aim_h) AS sum_aimed_headway_s,
           SUM(CAST((act_h < {BUNCH_RATIO} * aim_h) AS INT)) AS n_bunched,
           SUM(CAST((act_h > {GAP_RATIO} * aim_h) AS INT)) AS n_large_gap
    FROM h WHERE aim_h BETWEEN {HEADWAY_MIN_S} AND {HEADWAY_MAX_S}
    GROUP BY service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref"""))

## Alert impact
For every alert on a known line: calls in the same line **before**, **during** and **after** the alert.
Before/after windows are as long as the alert, clipped to 1–6 hours. Alerts without times, without a line,
or on a line that is not in stop_visits are skipped.

In [ ]:
alerts = read(SILVER_DIR, "alerts", required=False)
if alerts is not None and alerts.limit(1).count():
    al = alerts.select(
        F.monotonically_increasing_id().alias("alert_id"),
        opt(alerts, "situation_number").alias("situation_number"),
        opt(alerts, "line_ref").cast("string").alias("line_ref"),
        opt(alerts, "category").alias("category"),
        opt(alerts, "severity").alias("severity"),
        opt(alerts, "planned").alias("planned"),
        F.coalesce(to_utc(alerts, "valid_start"), to_utc(alerts, "first_seen")).alias("s"),
        F.coalesce(to_utc(alerts, "valid_end"), to_utc(alerts, "last_seen")).alias("e"))
    n_alerts = al.count()
    known = sv.select("line_ref").distinct()
    usable = (al.where(F.col("s").isNotNull() & F.col("e").isNotNull() & F.col("line_ref").isNotNull())
              .join(known, "line_ref", "left_semi"))
    n_usable = usable.count()

    dur_us = (F.least(F.greatest(F.expr(SECS.format(a="e", b="s")), F.lit(ALERT_MIN_WINDOW_S)),
                      F.lit(ALERT_MAX_WINDOW_S)) * 1e6).cast("long")
    usable = usable.withColumn("dur_us", dur_us)
    minus = lambda ts: F.timestamp_micros(F.unix_micros(F.col(ts)) - F.col("dur_us"))
    plus = lambda ts: F.timestamp_micros(F.unix_micros(F.col(ts)) + F.col("dur_us"))
    keep = ["alert_id", "situation_number", "line_ref", "category", "severity", "planned"]
    windows = (usable.select(*keep, F.lit(0).alias("p"), F.lit("before").alias("period"),
                             minus("s").alias("window_start"), F.col("s").alias("window_end"))
               .unionByName(usable.select(*keep, F.lit(1).alias("p"), F.lit("during").alias("period"),
                                          F.col("s").alias("window_start"), F.col("e").alias("window_end")))
               .unionByName(usable.select(*keep, F.lit(2).alias("p"), F.lit("after").alias("period"),
                                          F.col("e").alias("window_start"), plus("e").alias("window_end"))))

    calls = sv.select(F.col("line_ref").alias("c_line"), F.col("aimed_time").alias("t"),
                      "cancelled", "valid", "delay_s")
    joined = windows.join(calls, (F.col("line_ref") == F.col("c_line")) & (F.col("t") >= F.col("window_start"))
                          & (F.col("t") < F.col("window_end")), "left")
    v = F.coalesce(F.col("valid"), F.lit(False))
    n_if = lambda cond: F.sum(F.when(cond, 1).otherwise(0))
    impact = (joined.groupBy(*keep, "p", "period", "window_start", "window_end")
              .agg(F.count("t").alias("n_calls"),
                   n_if(F.coalesce(F.col("cancelled"), F.lit(False))).alias("n_cancelled"),
                   n_if(v).alias("n_valid"),
                   n_if(v & (F.col("delay_s") < EARLY_S)).alias("n_early"),
                   n_if(v & (F.col("delay_s") >= EARLY_S) & (F.col("delay_s") < LATE_S)).alias("n_on_time"),
                   n_if(v & (F.col("delay_s") >= LATE_S)).alias("n_late"),
                   F.coalesce(F.sum(F.when(v, F.col("delay_s"))), F.lit(0.0)).alias("sum_delay_s"))
              .orderBy("alert_id", "p")
              .drop("alert_id", "p"))
    print(f"  alerts: {n_usable} usable of {n_alerts} ({n_alerts - n_usable} skipped: "
          "no times, no line_ref, or line not in stop_visits)")
    emit("gold_alert_impact", impact)

## Vehicle positions

In [ ]:
vp_raw = read(SILVER_DIR, "vehicle_positions")
vp = vp_raw.withColumns({
    "snapshot_time": to_utc(vp_raw, "snapshot_time"),
    "is_stale": to_bool(vp_raw, "is_stale"),
    "age_s": num(vp_raw, "age_s"),
    "delay": num(vp_raw, "delay"),
    "line_ref": label(vp_raw, "line_ref"),
    "vehicle_id": label(vp_raw, "vehicle_id"),
    "occupancy_status": label(vp_raw, "occupancy_status", also_empty=True),
    "latitude": num(vp_raw, "latitude"),
    "longitude": num(vp_raw, "longitude"),
})
lv = local("snapshot_time")
vp = vp.withColumns({"local_date": F.date_format(lv, "yyyy-MM-dd"),
                     "local_hour": F.coalesce(F.hour(lv), F.lit(-1))}).cache()
vp.createOrReplaceTempView("vp")
r = vp.agg(F.count("*").alias("n"), F.avg(F.col("delay").isNotNull().cast("int")).alias("d")).first()
n_vp = r["n"]
print(f"vehicle_positions: {n_vp:,} rows; delay present in {(r['d'] or 0):.1%}")

emit("gold_fleet_activity", q("""
    SELECT local_date, local_hour, line_ref, COUNT(*) AS n_pings, SUM(CAST(is_stale AS INT)) AS n_stale,
           COALESCE(SUM(age_s), 0) AS sum_age_s, COUNT(age_s) AS n_age,
           COALESCE(SUM(delay), 0) AS sum_delay, COUNT(delay) AS n_delay,
           COUNT(DISTINCT vehicle_id) AS n_vehicles_at_this_grain   -- NOT additive across hours or lines
    FROM vp GROUP BY local_date, local_hour, line_ref"""))
emit("gold_vehicle_hour", q("""
    SELECT local_date, local_hour, line_ref, vehicle_id, COUNT(*) AS n_pings
    FROM vp GROUP BY local_date, local_hour, line_ref, vehicle_id"""))
emit("gold_occupancy", q("""
    SELECT local_date, local_hour, line_ref, occupancy_status, COUNT(*) AS n_pings
    FROM vp WHERE NOT is_stale GROUP BY local_date, local_hour, line_ref, occupancy_status"""))
emit("gold_position_grid", q(f"""
    SELECT local_date, local_hour, line_ref, gx, gy, CAST((gx + 0.5) * {GRID_DEG} AS DOUBLE) AS cell_lon, CAST((gy + 0.5) * {GRID_DEG} AS DOUBLE) AS cell_lat,
           COUNT(*) AS n_pings, COUNT(delay) AS n_delay, COALESCE(SUM(delay), 0) AS sum_delay
    FROM (SELECT *, CAST(floor(longitude / {GRID_DEG}) AS INT) AS gx, CAST(floor(latitude / {GRID_DEG}) AS INT) AS gy
          FROM vp WHERE NOT is_stale AND latitude IS NOT NULL AND longitude IS NOT NULL)
    GROUP BY local_date, local_hour, line_ref, gx, gy"""))

## v6: stop evidence (optional)

In [ ]:
ev_raw = read(V6_DIR, "stop_gps_status", required=False)
ev = None
if ev_raw is not None:
    ev = ev_raw.withColumns({
        "visit_time": to_utc(ev_raw, "visit_time"),
        "line_ref": F.col("route_id").cast("string"),
        "stop_ref": F.col("stop_id").cast("string"),
        "service_date": F.col("service_date").cast("string"),
        "time_lost_s": num(ev_raw, "time_lost_s"),
        "p_excess": num(ev_raw, "p_excess"),
        "ts_offset_enter_s": num(ev_raw, "ts_offset_enter_s"),
        "ts_offset_leave_s": num(ev_raw, "ts_offset_leave_s"),
    })
    off_ok = ((F.abs("ts_offset_enter_s") <= OFFSET_CLIP_S) & (F.abs("ts_offset_leave_s") <= OFFSET_CLIP_S))
    ev = ev.withColumns({
        "local_hour": F.coalesce(F.hour(local("visit_time")), F.lit(-1)),
        "off_enter": F.when(off_ok, F.col("ts_offset_enter_s")),
        "off_leave": F.when(off_ok, F.col("ts_offset_leave_s")),
    }).withColumn("hour_band", hour_band(F.col("local_hour"))).cache()
    ev.createOrReplaceTempView("ev")
    emit("gold_stop_evidence", q(f"""
        SELECT service_date, hour_band, line_ref, stop_ref, gps_class, COUNT(*) AS n_calls,
               COUNT(time_lost_s) AS n_time_lost, COALESCE(SUM(time_lost_s), 0) AS sum_time_lost_s,
               SUM(CAST((p_excess <= 0.05) AS INT)) AS n_p_excess_le_05,
               {hist_cols('tl', 'time_lost_s', 'time_lost_s IS NOT NULL', TL_EDGES)},
               COUNT(off_enter) AS n_offset, COALESCE(SUM(off_enter), 0) AS sum_off_enter_s,
               COALESCE(SUM(off_enter * off_enter), 0) AS sumsq_off_enter,
               COALESCE(SUM(off_leave), 0) AS sum_off_leave_s, COALESCE(SUM(off_leave * off_leave), 0) AS sumsq_off_leave
        FROM ev GROUP BY service_date, hour_band, line_ref, stop_ref, gps_class"""))

nul = read(V6_DIR, "time_lost_null", required=False)
if nul is not None and nul.limit(1).count():
    x = num(nul, "time_lost_s")
    bucket = sum(F.when(x >= e, 1).otherwise(0) for e in TL_EDGES)   # = number of edges <= x
    counts = dict(nul.where(x.isNotNull()).groupBy(bucket.alias("b")).count().collect())
    emit("gold_time_lost_reference", spark.createDataFrame(
        [(f"tl{i}", int(counts.get(i, 0))) for i in range(len(TL_EDGES) + 1)],
        "bucket string, n_stop_free_intervals long"))

## Data quality (long format: date, source, metric, value)

In [ ]:
def long_format(df, key, source, rank, metrics):
    """metrics: list of (name, aggregate expression). Returns rows date, source, metric, value."""
    wide = df.where(F.col(key).isNotNull()).groupBy(key).agg(*[e.alias(n) for n, e in metrics])
    pairs = ", ".join(f"{i}, '{n}', CAST(`{n}` AS double)" for i, n in enumerate(n for n, _ in metrics))
    return wide.select(F.col(key).alias("date"), F.lit(source).alias("source"), F.lit(rank).alias("src_rank"),
                       F.expr(f"stack({len(metrics)}, {pairs}) AS (m_rank, metric, value)"))

share = lambda cond: F.avg(F.coalesce(cond, F.lit(False)).cast("int"))
parts = [long_format(sv, "service_date", "stop_visits", 0, [
    ("n_calls", F.count("*")),
    ("n_valid", F.sum(F.col("valid").cast("int"))),
    ("share_actual", share(F.col("is_actual") & ~F.col("cancelled"))),
    ("share_no_prediction", share(F.col("no_prediction"))),
    ("share_outlier", share(F.col("is_actual") & F.col("delay_outlier"))),
    ("share_cancelled", share(F.col("cancelled"))),
    ("hours_with_data", F.countDistinct("local_hour"))])]

vp_metrics = [("n_pings", F.count("*")), ("share_stale", share(F.col("is_stale"))),
              ("n_vehicles", F.countDistinct("vehicle_id"))]
vp_dq = vp
if "last_updated" in vp_raw.columns:
    # share of pings that repeat an earlier (vehicle_id, last_updated) pair of the same day
    first = F.row_number().over(Window.partitionBy("local_date", "vehicle_id", "last_updated").orderBy(F.lit(1)))
    vp_dq = vp.withColumn("is_dup", first > 1)
    vp_metrics.append(("share_duplicate_pings", share(F.col("is_dup"))))
else:
    vp_metrics.append(("share_duplicate_pings", F.lit(None).cast("double")))
vp_metrics += [("hours_with_data", F.countDistinct("local_hour")),
               ("share_missing_position", share(F.col("latitude").isNull()))]
parts.append(long_format(vp_dq, "local_date", "vehicle_positions", 1, vp_metrics))

if ev is not None:
    parts.append(long_format(ev, "service_date", "v6_evidence", 2, [
        ("n_calls", F.count("*")),
        ("share_with_time_lost", share(F.col("time_lost_s").isNotNull())),
        ("share_no_vehicle", share(F.col("gps_class") == "no_vehicle")),
        ("median_off_enter_s", F.expr("percentile(off_enter, 0.5)")),
        ("median_off_leave_s", F.expr("percentile(off_leave, 0.5)"))]))

dq = parts[0]
for p in parts[1:]:
    dq = dq.unionByName(p)
emit("gold_data_quality", dq.orderBy("src_rank", "date", "m_rank").drop("src_rank", "m_rank"))

## Reconciliation
Every gold table must add up to its source.

In [ ]:
print("reconciliation")
n_buckets = len(DELAY_EDGES) + 1
check("visits cube holds every call", one("SELECT SUM(n_calls) FROM gold_visits_cube") == n_sv, f"{n_sv:,}")
check("stop cube holds every call", one("SELECT SUM(n_calls) FROM gold_stop_cube") == n_sv)
check("stop-order table holds every call", one("SELECT SUM(n_calls) FROM gold_stop_order_delay") == n_sv)
check("early + on time + late = valid",
      one("SELECT SUM(n_early + n_on_time + n_late - n_valid) FROM gold_visits_cube") == 0)
check("histogram buckets add up to valid",
      one(f"SELECT SUM({' + '.join(f'd{i}' for i in range(n_buckets))} - n_valid) FROM gold_visits_cube") == 0)
check("stop cube valid = visits cube valid",
      one("SELECT SUM(n_valid) FROM gold_stop_cube") == one("SELECT SUM(n_valid) FROM gold_visits_cube"))
cube_sum = one("SELECT SUM(sum_delay_s) FROM gold_visits_cube") or 0.0
row_sum = sv.where("valid").agg(F.sum("delay_s")).first()[0] or 0.0
check("sum of delays matches the row-level table",            # relative tolerance: Spark adds in parallel
      abs(cube_sum - row_sum) <= 1e-6 + 1e-12 * abs(row_sum), f"{row_sum:,.0f} s")
check("fleet table holds every ping", one("SELECT SUM(n_pings) FROM gold_fleet_activity") == n_vp)
check("occupancy table holds every fresh ping",
      one("SELECT SUM(n_pings) FROM gold_occupancy") == vp.where("NOT is_stale").count())
if ev is not None:
    n_ev = ev.count()
    check("evidence table holds every call", one("SELECT SUM(n_calls) FROM gold_stop_evidence") == n_ev, f"{n_ev:,}")

## Headline numbers and metadata

In [ ]:
h = q("SELECT SUM(n_valid) v, SUM(n_early) e, SUM(n_on_time) o, SUM(n_late) l, SUM(sum_delay_s) s "
      "FROM gold_visits_cube").first()
if h.v:
    print(f"headline: {int(h.v):,} valid calls | early {h.e / h.v:.1%} | on time {h.o / h.v:.1%} | "
          f"late {h.l / h.v:.1%} | mean delay {h.s / h.v:.0f} s (on time = {EARLY_S} s to {LATE_S} s, departure delay)")
    p50, p90, p95 = sv.where("valid").agg(F.expr("percentile(delay_s, array(0.5, 0.9, 0.95))")).first()[0]
    print(f"          exact delay percentiles: p50 {p50:.0f} s, p90 {p90:.0f} s, p95 {p95:.0f} s")
else:
    print("headline: no valid calls")

meta = {"built_at_utc": pd.Timestamp.now(tz="UTC").isoformat(), "engine": f"pyspark {spark.version}",
        "silver_dir": str(Path(SILVER_DIR).resolve()), "v6_dir": str(Path(V6_DIR).resolve()),
        "service_dates": ",".join(dates), "timezone": LOCAL_TZ, "delay_meaning": "departure delay (ET time behaves like departure)",
        "on_time_window_s": f"{EARLY_S}..{LATE_S}", "delay_edges_s": json.dumps(DELAY_EDGES), "time_lost_edges_s": json.dumps(TL_EDGES),
        "hour_bands": json.dumps(HOUR_BANDS), "grid_deg": str(GRID_DEG),
        "checks_passed": str(sum(c["ok"] for c in CHECKS)), "checks_failed": str(sum(not c["ok"] for c in CHECKS)),
        "failed_checks": json.dumps([c["check"] for c in CHECKS if not c["ok"]])}
meta.update({f"rows_{k}": str(v) for k, v in TABLES.items()})
emit("gold_meta", spark.createDataFrame(list(meta.items()), "key string, value string"))

## gold.duckdb for the dashboard (optional)
Copies every gold table into one DuckDB file, as the pandas version did. Skipped if `duckdb` is not installed.

In [ ]:
try:
    import duckdb
except ImportError:
    duckdb = None
    print("duckdb not installed: gold.duckdb skipped (the parquet tables are complete)")

if duckdb is not None:
    db_path = OUT / "gold.duckdb"
    db_path.unlink(missing_ok=True)
    con = duckdb.connect(str(db_path))
    for name in TABLES:
        con.execute(f"CREATE TABLE {name} AS SELECT * FROM read_parquet('{(OUT / (name + '.parquet')).as_posix()}/*.parquet')")
    con.close()
    print(f"gold.duckdb: {len(TABLES)} tables")

## Result

In [ ]:
failed = [c for c in CHECKS if not c["ok"]]
print(f"{'ALL CHECKS PASSED' if not failed else str(len(failed)) + ' CHECK(S) FAILED'} | gold written to {OUT.resolve()}")
assert not failed, f"failed checks: {[c['check'] for c in failed]}"